# 华泰动量类 exp_wgt_return_3m 因子复现

先前已经复现了exp_wgt_return_6m因子，在一定程度上取得了良好的效果，于是下面开始复现exp_wgt_return_3m因子，作为对比试验，首先仍然是直接用该因子构建一个简单的等权调仓策略，来检验裸用该因子选股时的效果

In [1]:
import math
import hashlib
import pandas as pd
from datetime import datetime, timedelta

import dai
from bigquant import bigtrader


# =====================================================
# 1. 策略参数区
# =====================================================

# 回测区间
START_DATE = "2024-01-01"
END_DATE = "2026-06-01"

# 每次持有的股票数量
HOLD_NUM = 100

# 每隔多少个交易日调仓一次
REBALANCE_DAYS = 20

# 目标总仓位：不要满仓，避免现金不足、手续费、整手约束造成回测不稳定
TARGET_TOTAL_WEIGHT = 0.98

# exp_wgt_return_3m：3个月改进动量因子
FACTOR_MONTHS = 3

# 3个月近似为63个交易日
LOOKBACK_DAYS = 21 * FACTOR_MONTHS

# 为了计算 m_lag，需要在回测开始日前多取一段历史
BEFORE_START_DAYS = 300

# list_days 是自然日口径，不是交易日口径
# 63个交易日大约对应91个自然日，这里多留缓冲
MIN_LIST_DAYS = int(LOOKBACK_DAYS * 365 / 252) + 30

# 初始资金
CAPITAL_BASE = 1_000_000

# 剔除行业：默认使用中信一级行业 cs_level1_name
# 本策略剔除：银行、国防军工、家电、商贸零售、建筑
EXCLUDE_INDUSTRIES = [
    "银行",
    "国防军工",
    "家电",
    "商贸零售",
    "建筑",
]


# =====================================================
# 2. 构造 exp_wgt_return_3m 因子表达式
# =====================================================

def lag(field: str, k: int) -> str:
    """生成 BigQuant DAI SQL 的 m_lag 表达式。k=0 表示当前值。"""
    if k == 0:
        return field
    return f"m_lag({field}, {k})"


num_terms = []
den_terms = []

for i in range(LOOKBACK_DAYS):
    # 指数衰减权重：exp(-x_i / N / 4)
    # N = FACTOR_MONTHS = 3
    decay_weight = math.exp(-i / FACTOR_MONTHS / 4.0)

    close_i = lag("close", i)
    close_i_1 = lag("close", i + 1)
    turn_i = lag("turn", i)

    # 第 t-i 日收益率
    ret_i = f"(({close_i} / NULLIF({close_i_1}, 0)) - 1.0)"

    # 分子：收益率 * 换手率 * 指数衰减权重
    num_terms.append(
        f"COALESCE(({decay_weight:.12g} * {turn_i} * {ret_i}), 0.0)"
    )

    # 分母：换手率 * 指数衰减权重
    den_terms.append(
        f"COALESCE(({decay_weight:.12g} * {turn_i}), 0.0)"
    )


num_expr = " + ".join(num_terms)
den_expr = " + ".join(den_terms)

industry_sql = ", ".join([f"'{x}'" for x in EXCLUDE_INDUSTRIES])

calc_start_date = (
    datetime.strptime(START_DATE, "%Y-%m-%d") - timedelta(days=BEFORE_START_DAYS)
).strftime("%Y-%m-%d")


# =====================================================
# 3. 用 DAI SQL 生成每日候选股票
# =====================================================

sql = f"""
WITH factor_raw AS (
    SELECT
        date,
        instrument,
        cs_level1_name,
        total_market_cap,
        list_days,
        is_risk_warning,
        suspended,

        -- 市值从小到大做截面百分位排名
        -- mcap_pct <= 1/3 表示市值排名后1/3，即小市值股票
        c_pct_rank(total_market_cap, ascending := true) AS mcap_pct,

        -- 华泰 exp_wgt_return_3m 因子
        ({num_expr}) / NULLIF(({den_expr}), 0) AS exp_wgt_return_3m

    FROM cn_stock_prefactors

    WHERE date >= '{calc_start_date}'
      AND date <= '{END_DATE}'

      -- 只取沪深A股，剔除北交所
      AND list_sector IN (1, 2, 3)

      -- 剔除ST / 风险警示
      AND is_risk_warning = 0

      -- 剔除停牌
      AND suspended = 0

      -- 剔除上市时间过短的股票
      AND list_days >= {MIN_LIST_DAYS}

      -- 基础字段非空
      AND close IS NOT NULL
      AND turn IS NOT NULL
      AND total_market_cap IS NOT NULL
      AND cs_level1_name IS NOT NULL
),

universe AS (
    SELECT
        *
    FROM factor_raw
    WHERE date >= '{START_DATE}'

      -- 市值排名后1/3
      AND mcap_pct <= 1.0 / 3.0

      -- 剔除指定中信一级行业
      AND cs_level1_name NOT IN ({industry_sql})

      -- 因子值非空
      AND exp_wgt_return_3m IS NOT NULL
),

ranked AS (
    SELECT
        date,
        instrument,
        cs_level1_name,
        total_market_cap,
        exp_wgt_return_3m,

        -- 稳定排序：
        -- 1. 因子值从低到高；
        -- 2. 因子值相同时，市值从小到大；
        -- 3. 仍相同时，股票代码从小到大。
        row_number() OVER (
            PARTITION BY date
            ORDER BY exp_wgt_return_3m ASC, total_market_cap ASC, instrument ASC
        ) AS factor_rank

    FROM universe
)

SELECT
    date,
    instrument,
    cs_level1_name,
    total_market_cap,
    exp_wgt_return_3m,
    factor_rank

FROM ranked

-- 每日只保留因子值最低的 HOLD_NUM 只股票
WHERE factor_rank <= {HOLD_NUM}

ORDER BY date ASC, factor_rank ASC, instrument ASC
"""

signal_df = dai.query(sql).df()

signal_df["date"] = pd.to_datetime(signal_df["date"]).dt.strftime("%Y-%m-%d")

print("原始每日信号样例：")
print(signal_df.head())
print("原始信号日期数量：", signal_df["date"].nunique())
print("原始信号股票数量：", signal_df["instrument"].nunique())

if signal_df.empty:
    raise ValueError("signal_df 为空，请检查日期范围、行业过滤、市值过滤或字段名称。")


# =====================================================
# 4. 按 REBALANCE_DAYS 生成调仓日
# =====================================================

all_signal_dates = sorted(signal_df["date"].unique())

# 每隔 REBALANCE_DAYS 个交易日调仓一次
rebalance_dates = set(all_signal_dates[::REBALANCE_DAYS])

signal_df = signal_df[signal_df["date"].isin(rebalance_dates)].copy()

# 为确保后续交易顺序稳定，先排序
signal_df = signal_df.sort_values(
    ["date", "factor_rank", "instrument"]
).reset_index(drop=True)

# 每个调仓日等权持有 HOLD_NUM 只股票
# 使用 TARGET_TOTAL_WEIGHT，避免满仓
signal_df["weight"] = signal_df.groupby("date")["instrument"].transform(
    lambda x: TARGET_TOTAL_WEIGHT / len(x)
)

print("调仓信号样例：")
print(signal_df.head())
print("调仓次数：", signal_df["date"].nunique())
print("调仓后涉及股票数量：", signal_df["instrument"].nunique())


# =====================================================
# 5. 打印信号哈希，用来检查信号是否稳定
# =====================================================

check_df = signal_df[["date", "instrument", "factor_rank", "weight"]].copy()
check_df = check_df.sort_values(["date", "factor_rank", "instrument"])

signal_hash = hashlib.md5(
    check_df.to_csv(index=False).encode("utf-8")
).hexdigest()

print("信号哈希：", signal_hash)


# =====================================================
# 6. BigTrader 回测函数
# =====================================================

def initialize(context: bigtrader.IContext):
    context.signal_data = context.data.copy()
    context.signal_data["date"] = pd.to_datetime(
        context.signal_data["date"]
    ).dt.strftime("%Y-%m-%d")

    context.signal_data = context.signal_data.sort_values(
        ["date", "factor_rank", "instrument"]
    ).reset_index(drop=True)

    context.signal_dates = set(context.signal_data["date"].unique())

    # 手续费设置：
    # buy_cost：买入佣金
    # sell_cost：卖出佣金 + 印花税近似合计
    # min_cost：最低手续费
    context.set_commission(
        bigtrader.PerOrder(
            buy_cost=0.0003,
            sell_cost=0.0013,
            min_cost=5.0
        )
    )


def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):
    today = data.current_dt.strftime("%Y-%m-%d")

    # 非调仓日不操作
    if today not in context.signal_dates:
        return

    today_df = context.signal_data[
        context.signal_data["date"] == today
    ].copy()

    if today_df.empty:
        return

    # 确保目标股票买入顺序稳定
    today_df = today_df.sort_values(
        ["factor_rank", "instrument"]
    ).reset_index(drop=True)

    target_instruments = set(today_df["instrument"].tolist())

    # 当前持仓
    current_positions = context.get_positions()
    holding_instruments = set(current_positions.keys())

    # 先卖出不在目标池里的股票
    # 注意：set 本身无序，所以这里必须 sorted，保证交易顺序稳定
    sell_list = sorted(holding_instruments - target_instruments)

    for instrument in sell_list:
        context.order_target_percent(instrument, 0)

    # 再买入 / 调整目标股票到等权
    # today_df 已经按照 factor_rank + instrument 排序，保证顺序稳定
    for _, row in today_df.iterrows():
        instrument = row["instrument"]
        target_weight = float(row["weight"])

        context.order_target_percent(
            instrument,
            target_weight
        )


# =====================================================
# 7. 运行回测
# =====================================================

instruments = sorted(signal_df["instrument"].unique().tolist())

performance = bigtrader.run(
    market=bigtrader.Market.CN_STOCK,
    frequency=bigtrader.Frequency.DAILY,

    start_date=START_DATE,
    end_date=END_DATE,

    capital_base=CAPITAL_BASE,
    instruments=instruments,
    data=signal_df,

    initialize=initialize,
    handle_data=handle_data,

    benchmark="000300.SH",

    # T日产生信号，下一根K线开盘成交
    order_price_field_buy="open",
    order_price_field_sell="open",

    # 单只股票成交量限制
    # 如果你想进一步排查回测不稳定，可以临时改成 1.0
    volume_limit=0.025,
)

原始每日信号样例：
         date instrument cs_level1_name  total_market_cap  exp_wgt_return_3m  \
0  2024-01-02  688701.SH        电力及公用事业      1.270264e+09                0.0   
1  2024-01-02  688565.SH             机械      1.360515e+09                0.0   
2  2024-01-02  688096.SH        电力及公用事业      1.457510e+09                0.0   
3  2024-01-02  688215.SH             机械      1.476275e+09                0.0   
4  2024-01-02  600455.SH            计算机      1.593928e+09                0.0   

   factor_rank  
0            1  
1            2  
2            3  
3            4  
4            5  
原始信号日期数量： 581
原始信号股票数量： 2114
调仓信号样例：
         date instrument cs_level1_name  total_market_cap  exp_wgt_return_3m  \
0  2024-01-02  688701.SH        电力及公用事业      1.270264e+09                0.0   
1  2024-01-02  688565.SH             机械      1.360515e+09                0.0   
2  2024-01-02  688096.SH        电力及公用事业      1.457510e+09                0.0   
3  2024-01-02  688215.SH             机械      1.476

## 防御性策略构建

由以上测试可见，裸用该因子时会产生与exp_wgt_return_6m同样的大幅度回撤，于是参照exp_wgt_return_6m的防御性策略，在这里同样给当前的因子构建同样的防御性策略来提高收益

In [1]:
import math
import hashlib
import ast
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import dai
from bigquant import bigtrader
from IPython.display import display


# =====================================================
# 1. 策略参数区
# =====================================================

START_DATE = "2020-01-01"
END_DATE = "2022-06-01"

# 每次持有股票数量
HOLD_NUM = 100

# 每隔多少个交易日重新选股一次
REBALANCE_DAYS = 20

# 强势市场目标总仓位
TARGET_TOTAL_WEIGHT = 0.98

# 弱势市场防御总仓位
# 如果想弱势时完全空仓，改成 0.0
DEFENSIVE_TOTAL_WEIGHT = 0.30

# 趋势过滤指数
# 小市值策略建议用中证1000
TREND_INDEX = "000852.SH"

# 指数趋势均线天数
TREND_MA_DAYS = 20

# exp_wgt_return_3m：3个月改进动量因子
FACTOR_MONTHS = 3
LOOKBACK_DAYS = 21 * FACTOR_MONTHS

# 为了计算 m_lag 和指数均线，需要在回测开始日前多取一段历史
BEFORE_START_DAYS = max(300, TREND_MA_DAYS * 3)

# 为了计算下一交易日涨跌停，需要在结束日后多取一段数据
AFTER_END_DAYS = 30

# list_days 是自然日口径，不是交易日口径
MIN_LIST_DAYS = int(LOOKBACK_DAYS * 365 / 252) + 30

# 初始资金
CAPITAL_BASE = 1_000_000

# 剔除行业：默认使用中信一级行业 cs_level1_name
EXCLUDE_INDUSTRIES = [
    "银行",
    "国防军工",
    "家电",
    "商贸零售",
    "建筑",
]

# 回测基准
BENCHMARK = "000300.SH"

# 是否打印每日调仓 / 风控日志
VERBOSE = False

# 手续费参数，需要和 context.set_commission 保持一致
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COST = 5.0


# =====================================================
# 2. 构造 exp_wgt_return_3m 因子表达式
# =====================================================

def lag(field: str, k: int) -> str:
    """生成 BigQuant DAI SQL 的 m_lag 表达式。k=0 表示当前值。"""
    if k == 0:
        return field
    return f"m_lag({field}, {k})"


num_terms = []
den_terms = []

for i in range(LOOKBACK_DAYS):
    decay_weight = math.exp(-i / FACTOR_MONTHS / 4.0)

    close_i = lag("close", i)
    close_i_1 = lag("close", i + 1)
    turn_i = lag("turn", i)

    ret_i = f"(({close_i} / NULLIF({close_i_1}, 0)) - 1.0)"

    num_terms.append(
        f"COALESCE(({decay_weight:.12g} * {turn_i} * {ret_i}), 0.0)"
    )

    den_terms.append(
        f"COALESCE(({decay_weight:.12g} * {turn_i}), 0.0)"
    )


num_expr = " + ".join(num_terms)
den_expr = " + ".join(den_terms)

industry_sql = ", ".join([f"'{x}'" for x in EXCLUDE_INDUSTRIES])

calc_start_date = (
    datetime.strptime(START_DATE, "%Y-%m-%d") - timedelta(days=BEFORE_START_DAYS)
).strftime("%Y-%m-%d")

calc_end_date = (
    datetime.strptime(END_DATE, "%Y-%m-%d") + timedelta(days=AFTER_END_DAYS)
).strftime("%Y-%m-%d")


# =====================================================
# 3. 查询：每日候选目标股票 + 每日指数趋势信号
# =====================================================

signal_sql = f"""
WITH index_raw AS (
    SELECT
        date,
        instrument,
        close AS trend_index_close,

        AVG(close) OVER (
            PARTITION BY instrument
            ORDER BY date
            ROWS BETWEEN {TREND_MA_DAYS - 1} PRECEDING AND CURRENT ROW
        ) AS trend_index_ma

    FROM cn_stock_index_bar1d

    WHERE instrument = '{TREND_INDEX}'
      AND date >= '{calc_start_date}'
      AND date <= '{END_DATE}'
),

index_trend AS (
    SELECT
        date,
        trend_index_close,
        trend_index_ma,

        CASE
            WHEN trend_index_ma IS NULL THEN 0
            WHEN trend_index_close > trend_index_ma THEN 1
            ELSE 0
        END AS risk_on,

        CASE
            WHEN trend_index_ma IS NULL THEN {DEFENSIVE_TOTAL_WEIGHT}
            WHEN trend_index_close > trend_index_ma THEN {TARGET_TOTAL_WEIGHT}
            ELSE {DEFENSIVE_TOTAL_WEIGHT}
        END AS trend_total_weight

    FROM index_raw
),

factor_raw AS (
    SELECT
        date,
        instrument,
        cs_level1_name,
        total_market_cap,
        list_days,
        is_risk_warning,
        suspended,

        close,
        turn,

        c_pct_rank(total_market_cap, ascending := true) AS mcap_pct,

        ({num_expr}) / NULLIF(({den_expr}), 0) AS exp_wgt_return_3m

    FROM cn_stock_prefactors

    WHERE date >= '{calc_start_date}'
      AND date <= '{END_DATE}'

      -- 只取沪深A股，剔除北交所
      AND list_sector IN (1, 2, 3)

      -- 剔除ST / 风险警示
      AND is_risk_warning = 0

      -- 剔除停牌
      AND suspended = 0

      -- 剔除上市时间过短
      AND list_days >= {MIN_LIST_DAYS}

      AND close IS NOT NULL
      AND turn IS NOT NULL
      AND total_market_cap IS NOT NULL
      AND cs_level1_name IS NOT NULL
),

universe AS (
    SELECT
        fr.*,
        it.trend_index_close,
        it.trend_index_ma,
        it.risk_on,
        it.trend_total_weight

    FROM factor_raw fr

    LEFT JOIN index_trend it
        ON fr.date = it.date

    WHERE fr.date >= '{START_DATE}'

      -- 小市值股票池：市值排名后1/3
      AND fr.mcap_pct <= 1.0 / 3.0

      -- 剔除指定中信一级行业
      AND fr.cs_level1_name NOT IN ({industry_sql})

      -- 因子值非空
      AND fr.exp_wgt_return_3m IS NOT NULL

      -- 指数趋势信号非空
      AND it.trend_total_weight IS NOT NULL
),

ranked AS (
    SELECT
        date,
        instrument,
        cs_level1_name,
        total_market_cap,
        exp_wgt_return_3m,

        trend_index_close,
        trend_index_ma,
        risk_on,
        trend_total_weight,

        row_number() OVER (
            PARTITION BY date
            ORDER BY exp_wgt_return_3m ASC, total_market_cap ASC, instrument ASC
        ) AS factor_rank

    FROM universe
)

SELECT
    date,
    instrument,
    cs_level1_name,
    total_market_cap,
    exp_wgt_return_3m,
    factor_rank,

    trend_index_close,
    trend_index_ma,
    risk_on,
    trend_total_weight

FROM ranked

WHERE factor_rank <= {HOLD_NUM}

ORDER BY date ASC, factor_rank ASC, instrument ASC
"""

target_daily_df = dai.query(signal_sql).df()
target_daily_df["date"] = pd.to_datetime(target_daily_df["date"]).dt.strftime("%Y-%m-%d")

if target_daily_df.empty:
    raise ValueError("target_daily_df 为空，请检查日期范围、指数代码、行业过滤、市值过滤或字段名称。")

print("每日目标信号样例：")
print(target_daily_df.head())
print("每日目标信号日期数量：", target_daily_df["date"].nunique())
print("每日目标信号股票数量：", target_daily_df["instrument"].nunique())


# =====================================================
# 4. 生成低频选股调仓日
# =====================================================

all_dates = sorted(target_daily_df["date"].unique())

rebalance_dates = all_dates[::REBALANCE_DAYS]
rebalance_dates_set = set(rebalance_dates)

target_rebalance_df = target_daily_df[
    target_daily_df["date"].isin(rebalance_dates_set)
].copy()

target_rebalance_df = target_rebalance_df.sort_values(
    ["date", "factor_rank", "instrument"]
).reset_index(drop=True)

if target_rebalance_df.empty:
    raise ValueError("target_rebalance_df 为空，请检查 REBALANCE_DAYS 或信号数据。")


# =====================================================
# 5. 构造每日持仓目标：
#    选股名单低频更新，但仓位每日跟随趋势信号调整
# =====================================================

daily_trend_df = target_daily_df.drop_duplicates("date")[
    [
        "date",
        "trend_index_close",
        "trend_index_ma",
        "risk_on",
        "trend_total_weight",
    ]
].copy()

date_to_idx = {d: i for i, d in enumerate(all_dates)}

daily_target_blocks = []

for i, rebalance_date in enumerate(rebalance_dates):
    start_idx = date_to_idx[rebalance_date]

    if i + 1 < len(rebalance_dates):
        end_idx = date_to_idx[rebalance_dates[i + 1]]
        active_dates = all_dates[start_idx:end_idx]
    else:
        active_dates = all_dates[start_idx:]

    rebalance_targets = target_rebalance_df[
        target_rebalance_df["date"] == rebalance_date
    ].copy()

    rebalance_targets = rebalance_targets[
        [
            "instrument",
            "cs_level1_name",
            "total_market_cap",
            "exp_wgt_return_3m",
            "factor_rank",
        ]
    ].copy()

    for current_date in active_dates:
        block = rebalance_targets.copy()
        block["date"] = current_date
        block["rebalance_date"] = rebalance_date
        block["is_target"] = 1
        daily_target_blocks.append(block)


daily_target_df = pd.concat(daily_target_blocks, ignore_index=True)

daily_target_df = daily_target_df.merge(
    daily_trend_df,
    on="date",
    how="left"
)

daily_target_df["stock_count"] = daily_target_df.groupby("date")["instrument"].transform("count")

daily_target_df["weight"] = (
    daily_target_df["trend_total_weight"] / daily_target_df["stock_count"]
)

daily_target_df = daily_target_df.sort_values(
    ["date", "factor_rank", "instrument"]
).reset_index(drop=True)

if daily_target_df.empty:
    raise ValueError("daily_target_df 为空，请检查每日目标构造逻辑。")


# =====================================================
# 6. 查询每日涨跌停交易状态
#    只查曾经进入过目标池的股票，降低数据量
# =====================================================

all_target_instruments = sorted(daily_target_df["instrument"].unique().tolist())
instrument_sql = ", ".join([f"'{x}'" for x in all_target_instruments])

trade_status_sql = f"""
WITH trade_raw AS (
    SELECT
        date,
        instrument,

        is_risk_warning,
        suspended,

        open,
        upper_limit,
        lower_limit,

        m_lead(open, 1) AS next_open,
        m_lead(upper_limit, 1) AS next_upper_limit,
        m_lead(lower_limit, 1) AS next_lower_limit,
        m_lead(suspended, 1) AS next_suspended

    FROM cn_stock_prefactors

    WHERE date >= '{START_DATE}'
      AND date <= '{calc_end_date}'

      AND list_sector IN (1, 2, 3)

      AND instrument IN ({instrument_sql})

      AND open IS NOT NULL
      AND upper_limit IS NOT NULL
      AND lower_limit IS NOT NULL
)

SELECT
    date,
    instrument,

    is_risk_warning,
    suspended,

    next_open,
    next_upper_limit,
    next_lower_limit,
    next_suspended,

    CASE
        WHEN next_suspended = 1 THEN 0
        WHEN next_open IS NULL THEN 0
        WHEN next_upper_limit IS NULL THEN 0
        WHEN next_open >= next_upper_limit THEN 0
        ELSE 1
    END AS can_buy_next_open,

    CASE
        WHEN next_suspended = 1 THEN 0
        WHEN next_open IS NULL THEN 0
        WHEN next_lower_limit IS NULL THEN 0
        WHEN next_open <= next_lower_limit THEN 0
        ELSE 1
    END AS can_sell_next_open

FROM trade_raw

WHERE date >= '{START_DATE}'
  AND date <= '{END_DATE}'

ORDER BY date ASC, instrument ASC
"""

trade_status_df = dai.query(trade_status_sql).df()
trade_status_df["date"] = pd.to_datetime(trade_status_df["date"]).dt.strftime("%Y-%m-%d")

if trade_status_df.empty:
    raise ValueError("trade_status_df 为空，请检查涨跌停字段或目标股票列表。")


# =====================================================
# 7. 合并每日目标仓位与每日交易状态
#    保留所有曾经入选过目标池的股票，用于非目标持仓的卖出判断
# =====================================================

signal_df = trade_status_df.merge(
    daily_target_df[
        [
            "date",
            "instrument",
            "rebalance_date",
            "cs_level1_name",
            "total_market_cap",
            "exp_wgt_return_3m",
            "factor_rank",
            "trend_index_close",
            "trend_index_ma",
            "risk_on",
            "trend_total_weight",
            "stock_count",
            "weight",
            "is_target",
        ]
    ],
    on=["date", "instrument"],
    how="left"
)

signal_df["is_target"] = signal_df["is_target"].fillna(0).astype(int)
signal_df["weight"] = signal_df["weight"].fillna(0.0)
signal_df["factor_rank"] = signal_df["factor_rank"].fillna(999999).astype(int)

# 非目标股票也补上每日趋势字段，方便统一判断
signal_df = signal_df.merge(
    daily_trend_df,
    on="date",
    how="left",
    suffixes=("", "_daily")
)

for col in ["trend_index_close", "trend_index_ma", "risk_on", "trend_total_weight"]:
    daily_col = f"{col}_daily"
    if daily_col in signal_df.columns:
        signal_df[col] = signal_df[col].combine_first(signal_df[daily_col])
        signal_df = signal_df.drop(columns=[daily_col])

signal_df = signal_df.sort_values(
    ["date", "is_target", "factor_rank", "instrument"],
    ascending=[True, False, True, True]
).reset_index(drop=True)

print("每日目标仓位样例：")
print(
    signal_df[signal_df["is_target"] == 1][
        [
            "date",
            "rebalance_date",
            "instrument",
            "factor_rank",
            "exp_wgt_return_3m",
            "risk_on",
            "trend_total_weight",
            "weight",
            "can_buy_next_open",
            "can_sell_next_open",
        ]
    ].head(30)
)

print("选股调仓次数：", len(rebalance_dates))
print("每日风控日期数量：", signal_df["date"].nunique())
print("曾经入选目标池股票数量：", len(all_target_instruments))

trend_summary = daily_trend_df.copy()
print("强势日期数量：", int((trend_summary["risk_on"] == 1).sum()))
print("防御日期数量：", int((trend_summary["risk_on"] == 0).sum()))


# =====================================================
# 8. 信号哈希
# =====================================================

check_df = daily_target_df[
    [
        "date",
        "rebalance_date",
        "instrument",
        "factor_rank",
        "risk_on",
        "trend_total_weight",
        "weight",
    ]
].copy()

check_df = check_df.sort_values(["date", "factor_rank", "instrument"])

signal_hash = hashlib.md5(
    check_df.to_csv(index=False).encode("utf-8")
).hexdigest()

print("信号哈希：", signal_hash)


# =====================================================
# 9. BigTrader 回测函数
# =====================================================

def _safe_int(x, default=0):
    try:
        if pd.isna(x):
            return default
        return int(x)
    except Exception:
        return default


def _safe_float(x, default=0.0):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def _can_buy(info):
    """
    下一交易日开盘涨停 / 停牌，则不能买。
    如果没有信息，保守起见不买。
    """
    if info is None:
        return False

    return _safe_int(info.get("can_buy_next_open", 0), 0) == 1


def _can_sell(info):
    """
    下一交易日开盘跌停 / 停牌，则不能卖。
    如果没有信息，则尝试卖出，避免持仓长期卡住。
    """
    if info is None:
        return True

    return _safe_int(info.get("can_sell_next_open", 1), 1) == 1


def initialize(context: bigtrader.IContext):
    context.signal_data = context.data.copy()

    context.signal_data["date"] = pd.to_datetime(
        context.signal_data["date"]
    ).dt.strftime("%Y-%m-%d")

    context.signal_data = context.signal_data.sort_values(
        ["date", "is_target", "factor_rank", "instrument"],
        ascending=[True, False, True, True]
    ).reset_index(drop=True)

    # 每个有趋势信号和交易状态的日期都允许进行每日仓位控制
    context.signal_dates = set(context.signal_data["date"].unique())

    # 记录上一次目标权重，用于判断这次是加仓还是减仓
    context.desired_weight_map = {}

    context.set_commission(
        bigtrader.PerOrder(
            buy_cost=BUY_COST,
            sell_cost=SELL_COST,
            min_cost=MIN_COST
        )
    )


def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):
    today = data.current_dt.strftime("%Y-%m-%d")

    # 没有信号的日期不操作
    if today not in context.signal_dates:
        return

    today_all_df = context.signal_data[
        context.signal_data["date"] == today
    ].copy()

    if today_all_df.empty:
        return

    info_map = today_all_df.set_index("instrument").to_dict("index")

    today_target_df = today_all_df[
        today_all_df["is_target"] == 1
    ].copy()

    today_target_df = today_target_df.sort_values(
        ["factor_rank", "instrument"]
    ).reset_index(drop=True)

    target_instruments = set(today_target_df["instrument"].tolist())

    if not today_target_df.empty:
        today_total_weight = _safe_float(today_target_df["trend_total_weight"].iloc[0], 0.0)
        today_risk_on = _safe_int(today_target_df["risk_on"].iloc[0], 0)
    else:
        today_total_weight = 0.0
        today_risk_on = 0

    if VERBOSE:
        print(
            f"{today} 每日风控：risk_on={today_risk_on}, "
            f"target_total_weight={today_total_weight:.2%}"
        )

    current_positions = context.get_positions()
    holding_instruments = set(current_positions.keys())

    # =================================================
    # 1）卖出当前已经不在目标池里的股票
    #    注意：这个动作每日都会尝试。
    #    如果某只股票前一天因为跌停卖不出，后续每天继续尝试卖出。
    # =================================================

    sell_list = sorted(holding_instruments - target_instruments)

    for instrument in sell_list:
        info = info_map.get(instrument)

        if not _can_sell(info):
            if VERBOSE:
                print(f"{today} 跳过卖出 {instrument}：下一交易日开盘跌停或停牌")
            continue

        context.order_target_percent(instrument, 0)
        context.desired_weight_map[instrument] = 0.0

    # =================================================
    # 2）处理目标股票
    #    选股名单只在调仓日更新；
    #    但 weight 会每天根据指数趋势变化。
    #
    #    如果目标权重上升：需要买入，要求 can_buy。
    #    如果目标权重下降：需要卖出，要求 can_sell。
    # =================================================

    for _, row in today_target_df.iterrows():
        instrument = row["instrument"]
        target_weight = float(row["weight"])

        info = info_map.get(instrument)

        can_buy = _can_buy(info)
        can_sell = _can_sell(info)

        prev_weight = float(context.desired_weight_map.get(instrument, 0.0))
        diff = target_weight - prev_weight

        # 权重几乎不变，不重复下单
        if abs(diff) < 1e-8:
            continue

        # 目标权重为0：清仓
        if target_weight <= 0:
            if not can_sell:
                if VERBOSE:
                    print(f"{today} 跳过清仓 {instrument}：下一交易日开盘跌停或停牌")
                continue

            context.order_target_percent(instrument, 0)
            context.desired_weight_map[instrument] = 0.0
            continue

        # 需要加仓 / 新开仓
        if diff > 0:
            if not can_buy:
                if VERBOSE:
                    print(f"{today} 跳过买入/加仓 {instrument}：下一交易日开盘涨停或停牌")
                continue

            context.order_target_percent(instrument, target_weight)
            context.desired_weight_map[instrument] = target_weight
            continue

        # 需要减仓
        if diff < 0:
            if not can_sell:
                if VERBOSE:
                    print(f"{today} 跳过减仓 {instrument}：下一交易日开盘跌停或停牌")
                continue

            context.order_target_percent(instrument, target_weight)
            context.desired_weight_map[instrument] = target_weight
            continue


# =====================================================
# 10. 运行回测
# =====================================================

instruments = sorted(all_target_instruments)

performance = bigtrader.run(
    market=bigtrader.Market.CN_STOCK,
    frequency=bigtrader.Frequency.DAILY,

    start_date=START_DATE,
    end_date=END_DATE,

    capital_base=CAPITAL_BASE,
    instruments=instruments,
    data=signal_df,

    initialize=initialize,
    handle_data=handle_data,

    benchmark=BENCHMARK,

    # T日产生信号，下一根K线开盘成交
    order_price_field_buy="open",
    order_price_field_sell="open",

    # 单只股票成交量限制
    volume_limit=0.025,
)


# =====================================================
# 11. 回测后统计：换手率、交易成本损耗、最大回撤区间
# =====================================================

def _get_raw_perf_df(performance):
    """
    兼容不同 BigQuant 环境下的 raw_perf 形式。

    有些环境：
        performance.raw_perf 是 DataFrame

    有些环境：
        performance.raw_perf 是对象，需要 read_df()
    """
    raw_perf_obj = performance.raw_perf

    if isinstance(raw_perf_obj, pd.DataFrame):
        return raw_perf_obj.copy()

    if hasattr(raw_perf_obj, "read_df"):
        return raw_perf_obj.read_df().copy()

    if isinstance(raw_perf_obj, dict):
        return pd.DataFrame(raw_perf_obj).copy()

    raise TypeError(
        f"无法识别 performance.raw_perf 的类型：{type(raw_perf_obj)}。"
        "请先运行 print(type(performance.raw_perf)) 和 print(performance.raw_perf) 检查。"
    )


def _as_list(x):
    """
    将 raw_perf 中的 transactions / orders 字段统一转成 list。
    """
    if x is None:
        return []

    if isinstance(x, float) and pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    if isinstance(x, tuple):
        return list(x)

    if isinstance(x, dict):
        return [x]

    if isinstance(x, str):
        s = x.strip()

        if s == "" or s.lower() in ["nan", "none", "null", "[]"]:
            return []

        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, list):
                return obj
            if isinstance(obj, dict):
                return [obj]
            return []
        except Exception:
            return []

    return []


def _get_first_value(d, keys, default=np.nan):
    """
    从交易记录 dict 中兼容读取字段。
    """
    if not isinstance(d, dict):
        return default

    for key in keys:
        if key in d and d[key] is not None:
            return d[key]

    return default


def _to_float(x, default=np.nan):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def _find_portfolio_value_col(raw_perf_df, capital_base):
    """
    自动寻找账户总资产字段。
    如果找不到，但存在 algorithm_period_return，则估算 portfolio_value。
    """
    candidate_cols = [
        "portfolio_value",
        "account_value",
        "total_value",
        "ending_value",
        "net_value",
        "portfolio_value_",
    ]

    for col in candidate_cols:
        if col in raw_perf_df.columns:
            return col

    if "algorithm_period_return" in raw_perf_df.columns:
        raw_perf_df["_portfolio_value_estimated"] = (
            capital_base * (1.0 + raw_perf_df["algorithm_period_return"].astype(float))
        )
        return "_portfolio_value_estimated"

    raise ValueError(
        "raw_perf 中找不到账户总资产字段。请先运行：\n"
        "print(raw_perf_df.columns.tolist())\n"
        "查看实际字段名。"
    )


def _parse_transaction_records(raw_perf_df):
    """
    将 raw_perf 中的 transactions 或 orders 展平成交易明细。
    优先使用 transactions，如果没有则尝试 orders。
    """

    if "transactions" in raw_perf_df.columns:
        record_col = "transactions"
    elif "orders" in raw_perf_df.columns:
        record_col = "orders"
    else:
        return pd.DataFrame(
            columns=[
                "date",
                "instrument",
                "amount",
                "price",
                "side",
                "trade_value",
                "transaction_cost",
                "cost_source",
            ]
        )

    records = []

    for dt, row in raw_perf_df.iterrows():
        tx_list = _as_list(row.get(record_col, []))

        for tx in tx_list:
            if not isinstance(tx, dict):
                continue

            instrument = _get_first_value(
                tx,
                ["instrument", "sid", "symbol", "asset", "order_book_id", "code"],
                default=""
            )

            amount = _to_float(
                _get_first_value(
                    tx,
                    ["amount", "qty", "quantity", "filled", "filled_quantity", "volume"],
                    default=0.0
                ),
                default=0.0
            )

            price = _to_float(
                _get_first_value(
                    tx,
                    ["price", "fill_price", "filled_price", "avg_price"],
                    default=np.nan
                ),
                default=np.nan
            )

            explicit_value = _to_float(
                _get_first_value(
                    tx,
                    ["trade_value", "turnover", "value", "money", "amount_value"],
                    default=np.nan
                ),
                default=np.nan
            )

            if pd.notna(explicit_value):
                trade_value = abs(explicit_value)
            elif pd.notna(price):
                trade_value = abs(amount * price)
            else:
                trade_value = np.nan

            side_raw = _get_first_value(
                tx,
                ["side", "direction", "action", "transaction_side"],
                default=""
            )

            side_str = str(side_raw).lower()

            if side_str in ["buy", "long", "open", "买入"]:
                side = "buy"
            elif side_str in ["sell", "short", "close", "卖出"]:
                side = "sell"
            else:
                side = "buy" if amount > 0 else "sell"

            actual_cost_candidates = [
                "commission",
                "transaction_cost",
                "cost",
                "fee",
                "fees",
                "tax",
                "stamp_tax",
                "total_cost",
            ]

            actual_cost_values = []
            for key in actual_cost_candidates:
                val = _to_float(tx.get(key, np.nan), default=np.nan)
                if pd.notna(val):
                    actual_cost_values.append(abs(val))

            if len(actual_cost_values) > 0:
                transaction_cost = sum(actual_cost_values)
                cost_source = "raw_perf"
            else:
                if pd.notna(trade_value):
                    rate = BUY_COST if side == "buy" else SELL_COST
                    transaction_cost = max(trade_value * rate, MIN_COST)
                    cost_source = "estimated"
                else:
                    transaction_cost = np.nan
                    cost_source = "unknown"

            records.append({
                "date": dt,
                "instrument": instrument,
                "amount": amount,
                "price": price,
                "side": side,
                "trade_value": trade_value,
                "transaction_cost": transaction_cost,
                "cost_source": cost_source,
            })

    tx_df = pd.DataFrame(records)

    if not tx_df.empty:
        tx_df["date"] = pd.to_datetime(tx_df["date"])

    return tx_df


def _calc_drawdown_interval(raw_perf_df, portfolio_value_col):
    """
    计算最大回撤和最大回撤区间。
    """
    portfolio_value = raw_perf_df[portfolio_value_col].astype(float).dropna()

    if portfolio_value.empty:
        raise ValueError(f"{portfolio_value_col} 为空，无法计算最大回撤区间。")

    net_value = portfolio_value / portfolio_value.iloc[0]
    running_max = net_value.cummax()
    drawdown = net_value / running_max - 1.0

    max_dd_end = drawdown.idxmin()
    max_drawdown = drawdown.loc[max_dd_end]

    max_dd_start = net_value.loc[:max_dd_end].idxmax()
    peak_value = net_value.loc[max_dd_start]
    trough_value = net_value.loc[max_dd_end]

    after_trough = net_value.loc[max_dd_end:]
    recovered = after_trough[after_trough >= peak_value]

    if len(recovered) > 0:
        recovery_date = recovered.index[0]
        recovered_flag = True
    else:
        recovery_date = pd.NaT
        recovered_flag = False

    return {
        "最大回撤": max_drawdown,
        "最大回撤开始日": max_dd_start,
        "最大回撤结束日": max_dd_end,
        "最大回撤修复日": recovery_date,
        "是否已修复": recovered_flag,
        "回撤峰值净值": peak_value,
        "回撤谷底净值": trough_value,
    }


def analyze_backtest_extra(performance, capital_base):
    """
    输出：
    1. 策略换手率
    2. 交易成本损耗
    3. 最大回撤区间
    """

    raw_perf = _get_raw_perf_df(performance)

    # 处理日期索引
    if not isinstance(raw_perf.index, pd.DatetimeIndex):
        if "period_open" in raw_perf.columns:
            raw_perf.index = pd.to_datetime(raw_perf["period_open"])
        elif "date" in raw_perf.columns:
            raw_perf.index = pd.to_datetime(raw_perf["date"])
        else:
            raw_perf.index = pd.to_datetime(raw_perf.index)

    raw_perf = raw_perf.sort_index()

    print("raw_perf 字段列表：")
    print(raw_perf.columns.tolist())

    portfolio_value_col = _find_portfolio_value_col(
        raw_perf,
        capital_base=capital_base
    )

    tx_df = _parse_transaction_records(raw_perf)

    portfolio_value = raw_perf[portfolio_value_col].astype(float)

    if tx_df.empty:
        total_trade_value = 0.0
        total_transaction_cost = 0.0
        trade_days = 0
        avg_daily_turnover = 0.0
        avg_trade_day_turnover = 0.0
        total_turnover = 0.0
        annualized_turnover = 0.0
        cost_source = "no_transaction_records"
    else:
        tx_df["trade_value"] = tx_df["trade_value"].fillna(0.0)
        tx_df["transaction_cost"] = tx_df["transaction_cost"].fillna(0.0)

        daily_trade_value = tx_df.groupby("date")["trade_value"].sum()
        daily_transaction_cost = tx_df.groupby("date")["transaction_cost"].sum()

        daily_stat = pd.DataFrame(index=raw_perf.index)
        daily_stat["portfolio_value"] = portfolio_value
        daily_stat["trade_value"] = daily_trade_value.reindex(daily_stat.index).fillna(0.0)
        daily_stat["transaction_cost"] = daily_transaction_cost.reindex(daily_stat.index).fillna(0.0)

        daily_stat["daily_turnover"] = (
            daily_stat["trade_value"] / daily_stat["portfolio_value"]
        ).replace([np.inf, -np.inf], np.nan).fillna(0.0)

        total_trade_value = daily_stat["trade_value"].sum()
        total_transaction_cost = daily_stat["transaction_cost"].sum()

        avg_portfolio_value = daily_stat["portfolio_value"].mean()

        total_turnover = (
            total_trade_value / avg_portfolio_value
            if avg_portfolio_value != 0
            else np.nan
        )

        days = len(daily_stat)
        years = days / 252
        annualized_turnover = total_turnover / years if years > 0 else np.nan

        avg_daily_turnover = daily_stat["daily_turnover"].mean()

        trade_days = int((daily_stat["trade_value"] > 0).sum())

        if trade_days > 0:
            avg_trade_day_turnover = daily_stat.loc[
                daily_stat["trade_value"] > 0,
                "daily_turnover"
            ].mean()
        else:
            avg_trade_day_turnover = 0.0

        cost_source = ",".join(
            sorted(tx_df["cost_source"].dropna().unique().tolist())
        )

    drawdown_info = _calc_drawdown_interval(
        raw_perf,
        portfolio_value_col=portfolio_value_col
    )

    final_portfolio_value = float(portfolio_value.iloc[-1])

    total_cost_over_initial = (
        total_transaction_cost / capital_base
        if capital_base != 0
        else np.nan
    )

    total_cost_over_final = (
        total_transaction_cost / final_portfolio_value
        if final_portfolio_value != 0
        else np.nan
    )

    summary = pd.DataFrame([
        ["总成交额", total_trade_value, "买入成交额 + 卖出成交额，按绝对值加总"],
        ["总换手率", total_turnover, "总成交额 / 回测期平均资产"],
        ["年化换手率", annualized_turnover, "总换手率 / 回测年数"],
        ["平均日换手率", avg_daily_turnover, "每日成交额 / 当日资产，然后取均值"],
        ["有交易日平均换手率", avg_trade_day_turnover, "只在有成交的日期上取平均"],
        ["有交易日期数", trade_days, "发生实际交易的日期数量"],
        ["交易成本合计", total_transaction_cost, f"成本来源：{cost_source}"],
        ["交易成本 / 初始资金", total_cost_over_initial, "交易成本合计 / 初始资金"],
        ["交易成本 / 期末资产", total_cost_over_final, "交易成本合计 / 期末资产"],
        ["最大回撤", drawdown_info["最大回撤"], "净值从峰值到谷底的最大跌幅"],
        ["最大回撤开始日", drawdown_info["最大回撤开始日"], "最大回撤对应的峰值日期"],
        ["最大回撤结束日", drawdown_info["最大回撤结束日"], "最大回撤对应的谷底日期"],
        ["最大回撤修复日", drawdown_info["最大回撤修复日"], "净值重新回到前高的日期；NaT表示截至回测结束未修复"],
        ["最大回撤是否已修复", drawdown_info["是否已修复"], ""],
    ], columns=["指标", "数值", "说明"])

    display_summary = summary.copy()

    percent_items = [
        "总换手率",
        "年化换手率",
        "平均日换手率",
        "有交易日平均换手率",
        "交易成本 / 初始资金",
        "交易成本 / 期末资产",
        "最大回撤",
    ]

    money_items = [
        "总成交额",
        "交易成本合计",
    ]

    def _format_value(row):
        name = row["指标"]
        value = row["数值"]

        if pd.isna(value):
            return ""

        if name in percent_items:
            return f"{float(value):.2%}"

        if name in money_items:
            return f"{float(value):,.2f}"

        if name in ["最大回撤开始日", "最大回撤结束日", "最大回撤修复日"]:
            try:
                return pd.to_datetime(value).strftime("%Y-%m-%d")
            except Exception:
                return str(value)

        if isinstance(value, (bool, np.bool_)):
            return "是" if value else "否"

        if isinstance(value, (int, np.integer)):
            return f"{value}"

        if isinstance(value, (float, np.floating)):
            return f"{value:.6f}"

        return str(value)

    display_summary["数值"] = display_summary.apply(_format_value, axis=1)

    print("\n" + "=" * 100)
    print("回测补充统计：换手率、交易成本损耗、最大回撤区间")
    print("=" * 100)

    display(display_summary)

    if not tx_df.empty:
        tx_preview = tx_df.head(20).copy()

        for col in ["trade_value", "transaction_cost"]:
            tx_preview[col] = tx_preview[col].map(
                lambda x: f"{x:,.2f}" if pd.notna(x) else ""
            )

        print("\n交易明细样例：")
        display(tx_preview)

    else:
        print("\n没有在 raw_perf 中解析到 transactions 或 orders 交易明细。")
        print("如果你想进一步计算真实换手和成本，需要检查 raw_perf 字段中是否存在交易记录字段。")

    return raw_perf, tx_df, summary


raw_perf_df, transaction_df, extra_summary = analyze_backtest_extra(
    performance=performance,
    capital_base=CAPITAL_BASE
)

每日目标信号样例：
         date instrument cs_level1_name  total_market_cap  exp_wgt_return_3m  \
0  2020-01-02  300362.SZ        电力及公用事业      1.219228e+09                0.0   
1  2020-01-02  300029.SZ       电力设备及新能源      1.240000e+09                0.0   
2  2020-01-02  300431.SZ             传媒      1.258784e+09                0.0   
3  2020-01-02  603269.SH             机械      1.282409e+09                0.0   
4  2020-01-02  002633.SZ             机械      1.284000e+09                0.0   

   factor_rank  trend_index_close  trend_index_ma  risk_on trend_total_weight  
0            1          5676.5579             NaN        0               0.30  
1            2          5676.5579             NaN        0               0.30  
2            3          5676.5579             NaN        0               0.30  
3            4          5676.5579             NaN        0               0.30  
4            5          5676.5579             NaN        0               0.30  
每日目标信号日期数量： 583
每日目标信号股票数量： 1